In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 27.0 MB/s eta 0:00:00


In [3]:
!python /content/drive/MyDrive/s1/run_stage1.py \
  --packet_csv /content/drive/MyDrive/s1/data/ar002_et12_20260511_001-stage1_packets.csv \
  --flow_csv /content/drive/MyDrive/s1/data/ar002_et12_20260511_001-stage1_flows.csv \
  --config /content/drive/MyDrive/s1/0708/0708C_ar002_et12_20260511_001/stage1_config_C.yaml \
  --out_dir /content/drive/MyDrive/s1/0708/0708C_ar002_et12_20260511_001/ \
  --export_embeddings \
  --use_supcon \
  --supcon_weight 0.05 \
  --supcon_temperature 0.10 \
  --supcon_warmup_epochs 5

[INFO] 随机种子已设置: 130
[INFO] device=cuda

[STEP 1] Building dataloaders...
[INFO] pipeline.py ------ build_dataloaders --- start
[INFO] pipeline.py ------ build_dataloaders --- split_method chronological
[INFO] data_io.py ------ read_stage1_csvs --- start
[data_io.py] --- read_stage1_csvs--- seed = 130, strategy = head, max_seq_len = 128
[INFO] data_io.py ------ read_stage1_csvs --- end
[INFO] splits.py ------ chronological_train_val_test_split-----------nominal_b1= 295329 nominal_b2= 337518
[INFO] splits.py ------ chronological_train_val_test_split-----------b1= 295329 b2= 337518
********** splits.py *****************_sanitize_three_way_boundaries: 边界安全修正，保证每个 split 至少有一个样本
[INFO] splits.py ------ chronological_train_val_test_split-----------train_df 
            flow_id  label  flow_start_timestamp_us
0  266605417401827      0         1778495176848505
1  266698416238684      0         1778495176848527
2  266751598479370      0         1778495176848539
[INFO] splits.py ------ chronologi

In [6]:
!python /content/drive/MyDrive/s2/run_stage2.py \
  --stage1_dir /content/drive/MyDrive/s1/0708/0708C_ar002_et12_20260511_001/ \
  --config /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/source_destination_attention/stage2_config_used.yaml \
  --out_dir /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/source_destination_attention/ws128_precision_regularized_v1 \
  --context_method source_destination \
  --context_policy online \
  --window_size 128 \
  --model_type source_destination_attention \
  --source_context_scale 0.2 \
  --destination_context_scale 0.1 \
  --gate_bias_init -2.0 \
  --source_gate_bias_init -2.0 \
  --destination_gate_bias_init -3.0 \
  --use_context_length_feature \
  --metric_for_best precision_label1 \
  --threshold_metric precision_lcb_at_recall \
  --threshold_recall_floor 0.80 \
  --threshold_precision_lcb_z 1.0 \
  --false_positive_penalty_weight 0.08 \
  --false_positive_penalty_margin 0.55 \
  --false_positive_topk_ratio 0.5 \
  --positive_margin_penalty_weight 0.02 \
  --positive_margin 0.45 \
  --sampler_pos_fraction 0.06 \
  --negative_alpha 1.0 \
  --positive_alpha 1.03 \
  --seed 130 \
  --epochs 500 \
  --batch_size 128

[INFO] Stage2 --------- seed: 130
[INFO] 随机种子已设置: 130
[INFO] Stage2 --------- device: cuda
[INFO] Stage2 --------- loading data... prepare_sorted_stage2_data
[INFO] Stage1 split:  train
[INFO] Stage1 split:  val
[INFO] Stage1 split:  test
[INFO] Stage2 --------- ContextIndexBuilder
[S2-ContextIndexBuilder]--context method: source_destination, window_size: 128, include_target: True, context_policy: online, endpoint_mode: same_endpoint, deduplicate: True
[INFO] Stage2 --------- context_indices =  [{'source': array([0]), 'destination': array([0])}, {'source': array([1]), 'destination': array([1])}, {'source': array([2]), 'destination': array([2])}]

[Dual Context Diagnostics] by split:

source_history_len:
          count       mean        std  min    50%    90%    95%    max
split                                                                 
test    84380.0  98.841088  47.513911  0.0  127.0  127.0  127.0  127.0
train  295329.0  77.019328  53.856065  0.0  106.0  127.0  127.0  127.0
val

In [7]:
!python /content/drive/MyDrive/s2/run_stage2.py \
  --stage1_dir /content/drive/MyDrive/s1/0708/0708C_ar002_et12_20260511_001/ \
  --config /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/target_query_gated-sampler_pos_fraction6-alpha11/stage2_config_used.yaml \
  --out_dir /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/target_query_gated-sampler_pos_fraction6-alpha11/ws128 \
  --context_method destination_host \
  --context_policy online \
  --window_size 128 \
  --model_type target_query_gated \
  --seed 130 \
  --epochs 500 \
  --batch_size 128

[INFO] Stage2 --------- seed: 130
[INFO] 随机种子已设置: 130
[INFO] Stage2 --------- device: cuda
[INFO] Stage2 --------- loading data... prepare_sorted_stage2_data
[INFO] Stage1 split:  train
[INFO] Stage1 split:  val
[INFO] Stage1 split:  test
[INFO] Stage2 --------- ContextIndexBuilder
[S2-ContextIndexBuilder]--context method: destination_host, window_size: 128, include_target: True, context_policy: online, endpoint_mode: same_endpoint, deduplicate: True
[INFO] Stage2 --------- context_indices =  [array([0]), array([1]), array([2])]

[Context Diagnostics] by split:
          count       mean        std  min    50%    90%    95%    max
split                                                                 
test    84380.0  89.765975  50.671856  0.0  127.0  127.0  127.0  127.0
train  295329.0  66.671583  54.095576  0.0   61.0  127.0  127.0  127.0
val     42189.0  87.496693  51.429910  0.0  127.0  127.0  127.0  127.0

[Context Diagnostics] by split and label:
                count       mean  

In [4]:
!python /content/drive/MyDrive/s2/run_stage2.py \
  --stage1_dir /content/drive/MyDrive/s1/0704/0704C_ar002_et12_20260511_001 \
  --config /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/target_query_gated-sampler_pos_fraction6-alpha11/stage2_config_used.yaml \
  --out_dir /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/target_query_gated-sampler_pos_fraction6-alpha11/ws128_0704 \
  --context_method source_host \
  --context_policy online \
  --window_size 128 \
  --model_type target_query_gated \
  --seed 130 \
  --epochs 500 \
  --batch_size 128

[INFO] Stage2 --------- seed: 130
[INFO] 随机种子已设置: 130
[INFO] Stage2 --------- device: cuda
[INFO] Stage2 --------- loading data... prepare_sorted_stage2_data
[INFO] Stage1 split:  train
[INFO] Stage1 split:  val
[INFO] Stage1 split:  test
[INFO] Stage2 --------- ContextIndexBuilder
[S2-ContextIndexBuilder]--context method: source_host, window_size: 128, include_target: True, context_policy: online, endpoint_mode: same_endpoint, deduplicate: True
[INFO] Stage2 --------- context_indices =  [array([0]), array([1]), array([2])]

[Context Diagnostics] by split:
          count       mean        std  min    50%    90%    95%    max
split                                                                 
test    84380.0  98.841088  47.513911  0.0  127.0  127.0  127.0  127.0
train  295329.0  77.019328  53.856065  0.0  106.0  127.0  127.0  127.0
val     42189.0  96.761431  48.540006  0.0  127.0  127.0  127.0  127.0

[Context Diagnostics] by split and label:
                count        mean      

In [8]:
!python /content/drive/MyDrive/s2/run_stage2.py \
  --stage1_dir /content/drive/MyDrive/s1/0704/0704C_ar002_et12_20260511_001 \
  --config /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/source_destination_attention/stage2_config_used.yaml \
  --out_dir /content/drive/MyDrive/s2/0709ar002_et12_20260511_001/source_destination_attention/ws128_precision_regularized_v1_0704C \
  --context_method source_destination \
  --context_policy online \
  --window_size 128 \
  --model_type source_destination_attention \
  --source_context_scale 0.2 \
  --destination_context_scale 0.1 \
  --gate_bias_init -2.0 \
  --source_gate_bias_init -2.0 \
  --destination_gate_bias_init -3.0 \
  --use_context_length_feature \
  --metric_for_best precision_label1 \
  --threshold_metric precision_lcb_at_recall \
  --threshold_recall_floor 0.80 \
  --threshold_precision_lcb_z 1.0 \
  --false_positive_penalty_weight 0.08 \
  --false_positive_penalty_margin 0.55 \
  --false_positive_topk_ratio 0.5 \
  --positive_margin_penalty_weight 0.02 \
  --positive_margin 0.45 \
  --sampler_pos_fraction 0.06 \
  --negative_alpha 1.0 \
  --positive_alpha 1.03 \
  --seed 130 \
  --epochs 500 \
  --batch_size 128

[INFO] Stage2 --------- seed: 130
[INFO] 随机种子已设置: 130
[INFO] Stage2 --------- device: cuda
[INFO] Stage2 --------- loading data... prepare_sorted_stage2_data
[INFO] Stage1 split:  train
[INFO] Stage1 split:  val
[INFO] Stage1 split:  test
[INFO] Stage2 --------- ContextIndexBuilder
[S2-ContextIndexBuilder]--context method: source_destination, window_size: 128, include_target: True, context_policy: online, endpoint_mode: same_endpoint, deduplicate: True
[INFO] Stage2 --------- context_indices =  [{'source': array([0]), 'destination': array([0])}, {'source': array([1]), 'destination': array([1])}, {'source': array([2]), 'destination': array([2])}]

[Dual Context Diagnostics] by split:

source_history_len:
          count       mean        std  min    50%    90%    95%    max
split                                                                 
test    84380.0  98.841088  47.513911  0.0  127.0  127.0  127.0  127.0
train  295329.0  77.019328  53.856065  0.0  106.0  127.0  127.0  127.0
val